In [1]:
import pandas as pd
import requests
import time
import pyalex
import re
from tqdm.auto import tqdm
from bs4 import BeautifulSoup
import cloudscraper

tqdm.pandas()
pyalex.config.email = "minhtruc103@gmail.com"

In [4]:
# --- CÁC HÀM LẤY DỮ LIỆU ---
def fetch_openalex(doi):
    try:
        return pyalex.Works[f"https://doi.org/{doi}"].get('abstract')
    except: return None

def fetch_semantic_scholar(doi):
    url = f"https://api.semanticscholar.org/graph/v1/paper/DOI:{doi}?fields=abstract"
    try:
        time.sleep(1.1)
        res = requests.get(url, timeout=10)
        if res.status_code == 200: return res.json().get('abstract')
    except: return None

def fetch_crossref(doi):
    url = f"https://api.crossref.org/works/{doi}"
    headers = {"User-Agent": "AcademicScraper/1.0 (mailto:minhtruc103@gmail.com)"}
    try:
        res = requests.get(url, headers=headers, timeout=10)
        if res.status_code == 200:
            abs_text = res.json().get('message', {}).get('abstract')
            if abs_text:
                return re.sub(r'<[^>]+>', '', abs_text).strip()
    except: return None

def fetch_beautifulsoup_safe(doi):
    """
    Sử dụng Cloudscraper và BeautifulSoup (Phiên bản an toàn chống False Positive)
    """
    url = f"https://doi.org/{doi}"

    # Khởi tạo Cloudscraper để đóng giả trình duyệt thật, vượt qua Cloudflare
    scraper = cloudscraper.create_scraper(
        browser={
            'browser': 'chrome',
            'platform': 'windows',
            'desktop': True
        }
    )

    try:
        # Thay requests.get bằng scraper.get
        response = scraper.get(url, timeout=20)
        if response.status_code != 200:
            return None

        soup = BeautifulSoup(response.text, 'html.parser')

        # Heuristics chống nhận diện sai
        invalid_keywords = ['open access', 'download', 'proceedings', 'cookie', 'subscribe', 'terms and conditions']
        def is_valid_abstract(text):
            if not text or len(text) < 150:
                return False
            text_lower = text.lower()
            for keyword in invalid_keywords:
                if keyword in text_lower and len(text) < 300:
                    return False
            return True

        # Chiến thuật 1: Meta Tag học thuật
        academic_meta_tags = [
            ('name', 'citation_abstract'),
            ('name', 'dc.description'),
            ('name', 'eprints.abstract'),
            ('property', 'og:description')
        ]
        for attr, value in academic_meta_tags:
            tag = soup.find('meta', attrs={attr: value})
            if tag and tag.get('content'):
                content = tag.get('content').strip()
                if is_valid_abstract(content): return content

        # Chiến thuật 2: HTML Elements cụ thể
        html_elements = [
            ('id', 'Abs1-content'),
            ('class_', 'c-article-section__content'),
            ('id', 'abstracts'),
            ('class_', 'abstract-section')
        ]
        for attr, value in html_elements:
            element = soup.find(id=value) if attr == 'id' else soup.find(class_=value)
            if element:
                content = element.get_text(separator=' ', strip=True)
                if is_valid_abstract(content): return content

        # Chiến thuật 3: Dò tìm tiêu đề H2/H3/Strong
        for header in soup.find_all(['h2', 'h3', 'strong']):
            if header.text and 'abstract' in header.text.lower():
                next_element = header.find_next_sibling('p')
                if next_element:
                    content = next_element.get_text(separator=' ', strip=True)
                    if is_valid_abstract(content): return content

    except Exception as e:
        pass

    return None

def fetch_with_llm_ollama(doi):
    # (Đã ẩn đi theo yêu cầu, bạn có thể uncomment nếu muốn chạy tiếp Ollama cho các bài siêu khó)
    pass

# --- HÀM TỔNG HỢP (TRẢ VỀ TUPLE) ---
def get_abstract_and_source(row):
    doi = row.get('doi')
    if pd.isna(doi) or str(doi).strip() == "":
        return None, "No DOI"

    # Chuỗi ưu tiên xử lý
    abstract = fetch_openalex(doi)
    if abstract: return abstract, "OpenAlex"

    abstract = fetch_semantic_scholar(doi)
    if abstract: return abstract, "Semantic Scholar"

    abstract = fetch_crossref(doi)
    if abstract: return abstract, "Crossref"

    # Đưa BeautifulSoup bọc thép vào đây
    abstract = fetch_beautifulsoup_safe(doi)
    if abstract: return abstract, "Web Scraping (Cloudscraper)"

    return None, "Failed"

# --- QUY TRÌNH XỬ LÝ CHÍNH ---
def process_data(input_file, output_file, id_map_file):
    print("Đang đọc file...")
    df = pd.read_csv(input_file)

    print("Đang cào dữ liệu (OpenAlex -> SemScholar -> Crossref -> Web Scraping)...")
    df[['abstract', 'source']] = df.progress_apply(get_abstract_and_source, axis=1, result_type='expand')

    print("\n--- BÁO CÁO NGUỒN LẤY ABSTRACT ---")
    print(df['source'].value_counts())

    df_map = df[['id', 'doi', 'source']].copy()
    df_map.to_csv(id_map_file, index=False)
    print(f"\nĐã lưu danh sách mapping tại: {id_map_file}")

    failed_df = df[df['source'] == 'Failed']
    print(f"[CẢNH BÁO] Còn lại {len(failed_df)} bài báo KHÔNG lấy được abstract.")

    df_final = df.drop(columns=['source'])
    df_final.to_csv(output_file, index=False)
    print(f"Đã lưu dữ liệu chính tại: {output_file}")

In [ ]:
process_data('data/test (2).csv', 'data/test.csv', 'data/id_source_map1.csv')
process_data('data/Stage_1_publcitrain.csv', 'data/train.csv', 'data/train_id_source_map.csv')

Đang đọc file...
Đang cào dữ liệu (OpenAlex -> SemScholar -> Crossref -> Web Scraping)...


  0%|          | 0/86 [00:00<?, ?it/s]


--- BÁO CÁO NGUỒN LẤY ABSTRACT ---
source
Semantic Scholar               43
Crossref                       23
Failed                         15
Web Scraping (Cloudscraper)     5
Name: count, dtype: int64

Đã lưu danh sách mapping tại: data/id_source_map1.csv
[CẢNH BÁO] Còn lại 15 bài báo KHÔNG lấy được abstract.
Đã lưu dữ liệu chính tại: data/test.csv
